# Full example

This script can be used as an example to start processing and analyzing your data. Only minor changes are needed to get going.

It loads data, does some preprocessing, extracts metrics, plots data, and saves it to a file.

We recommend reading up on the individual functions in the examples or API.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt

import piprkit

ps = piprkit.load_real_series()
ps.filters.rolling_mean(0.1)
ps.filters.limit_rate_of_change()
ps.drop_nan()
ps.add_blinks([(153.5, 156.5)])
ps.filters.remove_blinks()
ps.trim_size()

trials = ps.split(pre_stimulus_duration=10.0, post_stimulus_duration=55.0)

rows = []
fig, axes = plt.subplots(len(trials), 1, figsize=(8, 2.5 * len(trials)), sharex=True, sharey=True)
for number, (trial, ax) in enumerate(zip(trials, axes), start=1):
    baseline = trial.metrics.baseline(duration=5.0)
    trial.filters.apply_baseline_correction(duration=5.0)
    model = piprkit.PhaseModel(trial, baseline_duration=5.0)

    results = [
        baseline,
        trial.metrics.peak_constriction(),
        trial.metrics.time_to_peak(),
        trial.metrics.pipr_6s(),
        trial.metrics.auc_early(baseline_duration=5.0),
        model.metrics.constriction_v(),
        model.metrics.redilation_v(),
        model.metrics.plateau_pipr(),
    ]
    rows += piprkit.metric_rows(results, participant="example", trial=number)

    model.plot(ax=ax, scatter=True, s=2, color="grey", label=f"Trial {number}")
    model.plot_phases(ax=ax)
    model.plot_light_stimulus(ax=ax)
    ax.legend(fontsize=7, loc="lower right")
axes[-1].set_xlabel("Time (s)")
plt.show()

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "metrics.csv"
    piprkit.save_metric_rows(rows, path)
    print(path.read_text())